# Project 06: Diffusion Math and Scheduler Deep Dive

Analyze linear and cosine diffusion schedules on real image data,
visualize forward diffusion, and capture reproducibility fingerprints.


## Dataset Source and Download Instructions

- Primary dataset: CelebA
- Official source: https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html
- License: check official CelebA terms
- Auto-path in notebook: load local CelebA files if already available
- Fallback real dataset (auto-download): CIFAR-10 from https://www.cs.toronto.edu/~kriz/cifar.html
- Manual fallback:
  1. Download CelebA from official source and place in `./data/celeba`.
  2. If unavailable, use CIFAR-10 fallback cell as-is.


In [ ]:
%pip -q install neurogebra torch torchvision matplotlib


In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torchvision import datasets, transforms

from neurogebra.logging.fingerprint import TrainingFingerprint

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
data_root = Path("./data")
transform_face = transforms.Compose([
    transforms.CenterCrop(140),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])
transform_cifar = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

source_used = "CelebA"
try:
    ds = datasets.CelebA(root=data_root, split="train", target_type="attr", transform=transform_face, download=False)
    if len(ds) == 0:
        raise RuntimeError("CelebA dataset appears empty")
except Exception as exc:
    print("CelebA not ready, switching to CIFAR-10 fallback:", exc)
    ds = datasets.CIFAR10(root=data_root, train=True, transform=transform_cifar, download=True)
    source_used = "CIFAR-10 fallback"

sample_imgs = torch.stack([ds[i][0] if isinstance(ds[i], tuple) else ds[i] for i in range(8)], dim=0)
sample_imgs = sample_imgs[:8]
print("Using dataset source:", source_used)
print("Sample tensor shape:", sample_imgs.shape)


In [ ]:
T = 1000

def linear_beta_schedule(timesteps, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, timesteps)

def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * torch.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 1e-5, 0.999)

betas_lin = linear_beta_schedule(T)
betas_cos = cosine_beta_schedule(T)

alpha_bar_lin = torch.cumprod(1 - betas_lin, dim=0)
alpha_bar_cos = torch.cumprod(1 - betas_cos, dim=0)

snr_lin = alpha_bar_lin / (1 - alpha_bar_lin)
snr_cos = alpha_bar_cos / (1 - alpha_bar_cos)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(betas_lin.numpy(), label="linear")
plt.plot(betas_cos.numpy(), label="cosine")
plt.title("Beta Schedule")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(alpha_bar_lin.numpy(), label="linear")
plt.plot(alpha_bar_cos.numpy(), label="cosine")
plt.title("Alpha Bar")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(snr_lin.numpy(), label="linear")
plt.plot(snr_cos.numpy(), label="cosine")
plt.yscale("log")
plt.title("SNR")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Forward diffusion visualization at selected timesteps
x0 = sample_imgs[0:1]
if x0.shape[1] == 3:
    x0_gray = x0.mean(dim=1, keepdim=True)
else:
    x0_gray = x0

timesteps_to_show = [0, 50, 150, 400, 800, 999]
noisy_versions = []

for t in timesteps_to_show:
    a_bar = alpha_bar_lin[t]
    noise = torch.randn_like(x0_gray)
    xt = torch.sqrt(a_bar) * x0_gray + torch.sqrt(1 - a_bar) * noise
    noisy_versions.append(xt[0, 0].numpy())

plt.figure(figsize=(14, 3))
for i, t in enumerate(timesteps_to_show):
    plt.subplot(1, len(timesteps_to_show), i + 1)
    plt.imshow(noisy_versions[i], cmap="gray")
    plt.title(f"t={t}")
    plt.axis("off")
plt.suptitle("Forward Diffusion (Linear Schedule)")
plt.tight_layout()
plt.show()


In [ ]:
fp = TrainingFingerprint.capture(
    model_info={"name": "diffusion_math_scheduler", "source": source_used},
    hyperparameters={"timesteps": T, "schedule": "linear+cosine"},
    dataset=sample_imgs.numpy(),
    random_seed=SEED,
)

print(fp.format_text())

out_dir = Path("./logs_diffusion_math")
out_dir.mkdir(parents=True, exist_ok=True)
with (out_dir / "fingerprint.json").open("w", encoding="utf-8") as f:
    json.dump(fp.to_dict(), f, indent=2)

print("Fingerprint saved to", out_dir / "fingerprint.json")
